In [158]:
import pandapower as pp
import torch
import numpy as np
import julia
julia.install()

from julia.api import Julia
jl = Julia(compiled_modules=False)

net = pp.from_pickle("/home/iboero/Tesis/crear_modelo_uy/GNN4OPF/modelos_pp/uy_pp_net_v14_(sin_eolico_ni_solar).p")
net.bus["pm_param/setpoint_v"] = 1.0

X_test = np.load(f'/home/iboero/Tesis/unsupervised_uru/GNN4OPF/data/reduru/test/input.npy')


[ Info: Julia version info


Julia Version 1.6.0
Commit f9720dc2eb (2021-03-24 12:55 UTC)
Platform Info:
  OS: Linux (x86_64-pc-linux-gnu)
      "Manjaro Linux"
  uname: Linux 6.6.30-2-MANJARO #1 SMP PREEMPT_DYNAMIC Wed May  8 17:46:43 UTC 2024 x86_64 unknown
  CPU: 13th Gen Intel(R) Core(TM) i5-13400F: 
                 speed         user         nice          sys         idle          irq
       #1-16  4100 MHz   69333540 s       1220 s    1236324 s  218295241 s     276239 s
       
  Memory: 62.63358688354492 GB (759.4921875 MB free)
  Uptime: 1.810056e6 sec
  Load Avg:  9.3  8.72  8.52
  WORD_SIZE: 64
  LIBM: libopenlibm
  LLVM: libLLVM-11.0.1 (ORCJIT, goldmont)
Environment:
  HOME = /home/iboero
  PATH = /home/iboero/miniconda3/envs/proy/bin:/home/iboero/.vscode-server/cli/servers/Stable-5437499feb04f7a586f677b155b039bc2b3669eb/server/bin/remote-cli:/home/iboero/miniconda3/envs/proy/bin:/home/iboero/miniconda3/condabin:/home/iboero/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/bin:/opt/cuda/bin:/opt/cuda/nsi

[ Info: Julia executable: /home/iboero/julia/bin/julia
[ Info: Trying to import PyCall...
┌ Info: PyCall is already installed and compatible with Python executable.
│ 
│ PyCall:
│     python: /home/iboero/miniconda3/envs/proy/bin/python
│     libpython: /home/iboero/miniconda3/envs/proy/lib/libpython3.11.so.1.0
│ Python:
│     python: /home/iboero/miniconda3/envs/proy/bin/python
└     libpython: 


In [159]:
bus_map = {net.bus.index[i]: i for i in range(len(net.bus))}

# That also works for a list of buses
def bus_pos(buses):
    try:
        return [bus_map[bus] for bus in buses]
    except:
        return bus_map[buses]

In [151]:
# net.line.name = net.bus["name"].iloc[bus_pos(net.line.from_bus)].values + "_to_" + net.bus["name"].iloc[bus_pos(net.line.to_bus)].values

In [160]:
idx_gens = bus_pos(net.gen.bus.values.astype(int))
idx_load = bus_pos(net.load.bus.values.astype(int))
idx_sgens = bus_pos(net.sgen.bus.loc[net.sgen.controllable==False].values.astype(int))

idx = 1300
net.load.loc[:,'p_mw'] = X_test[idx,idx_load,0] 
net.load.loc[:,'q_mvar'] = X_test[idx,idx_load,1]
net.gen.loc[:,'p_mw'] =  X_test[idx,idx_gens,2]
net.sgen[net.sgen['controllable'] == False].loc[:,'p_mw'] = X_test[idx,idx_sgens,3]

/tmp/ipykernel_1885323/828973807.py:9: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy




In [161]:
import csv

with open('buses.csv', mode='r') as file:
    reader = csv.reader(file)
    # remove first line
    next(reader)
    
    try:
        buses_coord = [(float(rows[2]),float(rows[3])) for rows in reader]
    except:
        pass
net.bus_geodata.x = [x[1] for x in buses_coord]
net.bus_geodata.y = [x[0] for x in buses_coord]
net.bus_geodata.index = net.bus.index

In [162]:
from copy import deepcopy
net_pert = deepcopy(net)
del_idxs = [6,7]
net_pert.line.drop(del_idxs, inplace=True)


pp.runpm_vstab(net)
pp.runpm_vstab(net_pert)


normal_net_volt = net.res_bus['vm_pu']
altered_net_volt = net_pert.res_bus['vm_pu']

cambio_volt = np.abs(normal_net_volt - altered_net_volt)
cambio_volt

no costs are given - overall generated power is minimized


no costs are given - overall generated power is minimized


101      1.279335e-03
4000     1.145803e-10
4004     5.816285e-03
90000    2.207271e-04
90080    6.723187e-03
90100    1.601386e-03
90160    2.285142e-03
90180    1.871391e-03
90200    1.767594e-03
90300    1.382775e-03
90480    4.457152e-04
90500    2.169971e-03
90700    1.499858e-04
92000    1.975208e-03
92020    2.496332e-03
92030    4.498956e-03
92040    1.637521e-03
92050    1.118527e-03
92060    1.153893e-03
92061    1.478132e-03
92070    4.603457e-04
92080    1.236768e-03
92090    1.042871e-03
92091    7.843277e-04
92100    1.370690e-03
92110    1.374615e-03
92111    1.170209e-03
92124    7.295315e-04
92131    1.122527e-03
92132    1.122527e-03
92141    1.054793e-03
92142    1.112888e-03
92180    1.337500e-03
92200    1.427177e-03
92210    1.173924e-03
92220    1.243131e-03
92230    1.243131e-03
92240    1.239666e-03
92270    1.251377e-03
92280    1.432722e-03
92300    1.315836e-03
92310    1.285317e-03
92320    1.268946e-03
92330    1.307779e-03
92340    1.318943e-03
92350    1

In [163]:
net_pert.bus_geodata

,x,y,coords
101,-54.938115,-34.955739,NaN
4000,-57.930320,-31.212332,NaN
4004,-58.327850,-32.672407,NaN
90000,-57.941349,-31.274842,NaN
90080,-58.130494,-32.665934,NaN
90100,-56.186113,-34.835373,NaN
90160,-56.541138,-34.749778,NaN
90180,-56.932703,-33.760833,NaN
90200,-56.235881,-34.826261,NaN
90300,-56.129463,-34.892016,NaN


In [165]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Crear una figura vacía
fig = go.Figure()

# Agregar las líneas primero
for i, row in net_pert.line.iterrows():
    print(i, row)
    from_bus = net_pert.bus_geodata.loc[row['from_bus']]
    to_bus = net_pert.bus_geodata.loc[row['to_bus']]
    
    fig.add_trace(go.Scattermapbox(
        lon=[from_bus['x'], to_bus['x']],
        lat=[from_bus['y'], to_bus['y']],
        mode='lines',
        line=dict(width=1, color='black'),
        showlegend=False,  # No mostrar en la leyenda
        hoverinfo='text',
        hovertext=f"From: {row['from_bus']}<br>To: {row['to_bus']}"
    ))

# Agregar los trafos
for i, row in net_pert.trafo.iterrows():
    from_bus = net_pert.bus_geodata.loc[row['hv_bus']]
    to_bus = net_pert.bus_geodata.loc[row['lv_bus']]
    
    fig.add_trace(go.Scattermapbox(
        lon=[from_bus['x'], to_bus['x']],
        lat=[from_bus['y'], to_bus['y']],
        mode='lines',
        line=dict(width=1, color='blue'),
        showlegend=False,  # No mostrar en la leyenda
        hoverinfo='text',
        hovertext=f"From: {row['hv_bus']}<br>To: {row['lv_bus']}"
    ))


# Agregar las líneas primero
for i,row in net.line.iloc[del_idxs].iterrows():
    from_bus = net.bus_geodata.loc[row['from_bus']]
    to_bus = net.bus_geodata.loc[row['to_bus']]
    
    fig.add_trace(go.Scattermapbox(
        lon=[from_bus['x'], to_bus['x']],
        lat=[from_bus['y'], to_bus['y']],
        mode='lines',
        line=dict(width=1, color='red'),
        showlegend=False,  # No mostrar en la leyenda
        hoverinfo='text',
        hovertext=f"From: {row['from_bus']}<br>To: {row['to_bus']}"
    ))

# Añadir la columna de cambio de voltaje a net_pert.bus_geodata
net_pert.bus_geodata['cambio_voltaje'] = cambio_volt

# Crear una columna de tamaño para los nodos, por ejemplo multiplicando el cambio de voltaje por un factor
net_pert.bus_geodata['size'] = 20  # Ajusta el factor según sea necesario

# Calcular el centro del mapa basado en las coordenadas de los puntos
center_lon = net_pert.bus_geodata['x'].mean()
center_lat = net_pert.bus_geodata['y'].mean()

# Agregar los puntos después de las líneas
fig.add_trace(go.Scattermapbox(
    lon=net_pert.bus_geodata['x'],
    lat=net_pert.bus_geodata['y'],
    mode='markers',
    marker=go.scattermapbox.Marker(
        size=net_pert.bus_geodata['size'],
        color=cambio_volt,
        colorscale='Reds',
        cmax=cambio_volt.max(),
        cmin=cambio_volt.min(),
        colorbar=dict(title='Cambio Voltaje')
    ),
    hoverinfo='text',
    hovertext=pd.concat((net_pert.bus["name"],net_pert.bus_geodata["cambio_voltaje"]),axis=1).apply(lambda row: f"Bus: {row['name']}<br>Cambio Voltaje: {row['cambio_voltaje']}", axis=1)
))

# Configuración del mapa
fig.update_layout(mapbox_style='carto-positron')  # Estilo de mapa blanco y negro sencillo
fig.update_layout(margin={'r':0, 't':50, 'l':0, 'b':10})
fig.update_layout(mapbox=dict(
    zoom=5,
    center=go.layout.mapbox.Center(lon=center_lon, lat=center_lat)
))

fig.show()



0 name                         None
std_type                     None
from_bus                      101
to_bus                      92740
length_km                     1.0
r_ohm_per_km             0.149565
x_ohm_per_km               0.2457
c_nf_per_km            162.408777
g_us_per_km                   0.0
max_i_ka                 0.431088
df                            1.0
parallel                        1
type                         None
in_service                   True
max_loading_percent         100.0
Name: 0, dtype: object
1 name                         None
std_type                     None
from_bus                      101
to_bus                      92740
length_km                     1.0
r_ohm_per_km             0.149565
x_ohm_per_km               0.2457
c_nf_per_km            162.408777
g_us_per_km                   0.0
max_i_ka                 0.431088
df                            1.0
parallel                        1
type                         None
in_service           

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform

# Ejemplo de valores de nodos antes y después
valores_antes = np.array([10, 20, 30, 40, 50])
valores_despues = np.array([12, 18, 33, 39, 47])

# Calcular TAD
tad = np.sum(np.abs(valores_antes - valores_despues))
print("Total Absolute Difference:", tad)

# Calcular AC
ac = np.mean(np.abs(valores_antes - valores_despues))
print("Average Change:", ac)

# Calcular Desviación Estándar del Cambio
sigma_c = np.std(np.abs(valores_antes - valores_despues))
print("Standard Deviation of Change:", sigma_c)

# Matriz de adyacencia del grafo (ejemplo simple)
matriz_adyacencia = np.array([
    [0, 1, 0, 0, 0],
    [1, 0, 1, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1],
    [0, 0, 0, 1, 0]
])

# Calcular distancias geográficas/topológicas desde el nodo afectado
distancias = np.sum(matriz_adyacencia, axis=1)

# Calcular correlación espacial
cambios = np.abs(valores_antes - valores_despues)
correlacion_espacial = np.corrcoef(distancias, cambios)[0, 1]
print("Spatial Correlation:", correlacion_espacial)


In [ ]:
net_pert.